In [1]:
# Import Libraries
import pandas as pd
import numpy as np
# from scipy.stats import pearsonr, spearmanr, norm
# from scipy.spatial.distance import euclidean
# from sklearn.preprocessing import MinMaxScaler
# import statsmodels.api as sm
# import statsmodels.formula.api as smf
from tools import *
import os
import re
import time
import gc
import glob
from collections import Counter
import random
random.seed(621)

## Change input and output paths here

In [ ]:
# Path pattern to match your datasets
file_pattern = r"Z:\Projects\EMA_Project\Data\EMOTE\data_downloads_HLAJBYIHRK_2025-05-30*.csv"

# Find all matching files
csv_files = glob.glob(file_pattern)

# Scratch
output_folder = "Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Scratch"
os.makedirs(output_folder, exist_ok=True)

# Scratch
results_folder = "Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Results"
os.makedirs(results_folder, exist_ok=True)

<>:8: SyntaxWarning: invalid escape sequence '\P'
<>:12: SyntaxWarning: invalid escape sequence '\P'
<>:8: SyntaxWarning: invalid escape sequence '\P'
<>:12: SyntaxWarning: invalid escape sequence '\P'
C:\Users\bfsch\AppData\Local\Temp\ipykernel_27428\1471258796.py:8: SyntaxWarning: invalid escape sequence '\P'
  output_folder = "Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Scratch"
C:\Users\bfsch\AppData\Local\Temp\ipykernel_27428\1471258796.py:12: SyntaxWarning: invalid escape sequence '\P'
  results_folder = "Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Results"


In [ ]:
# Part 1: Load in all datasets that have what we want

# Define the relevant variables
core_required = ["Date_Local", "dataset", "UUID", "Time_Local", "ANG_ES", "HAP_ES"]
sad_options = ["SAD_ES", "DEP_ES"]
relaxation_options = ["RLX_ES", "CALM_ES"]
regulation_options = ["DIST_ES", "SUPR_ES", "REAP_ES", "REAP1_ES"]

# Combine into full variable set of interest
all_vars = core_required + sad_options + relaxation_options + regulation_options

In [5]:
# Track qualifying files and summaries
qualifying_files = []
dataset_summaries = []

all_uuids = set()
total_emas = 0

for file in csv_files:
    try:
        df = pd.read_csv(file)

        # Check inclusion criteria
        df_columns = df.columns
        has_all_core = all(var in df_columns for var in core_required)
        has_sad = any(var in df_columns for var in sad_options)
        has_relaxation = any(var in df_columns for var in relaxation_options)
        has_regulation = any(var in df_columns for var in regulation_options)

        if has_all_core and has_sad and has_relaxation and has_regulation:
            qualifying_files.append(file)

            # Keep only relevant variables that exist in this dataset
            relevant_vars = [var for var in all_vars if var in df.columns]
            df_sub = df[relevant_vars].dropna()

            # Update total EMAs and UUIDs
            total_emas += df_sub.shape[0]
            all_uuids.update(df_sub['UUID'].unique())

            # Number of UUIDs and average number of responses per UUID
            num_uuids = df_sub['UUID'].nunique()
            avg_responses_per_uuid = df_sub.shape[0] / num_uuids if num_uuids > 0 else 0

            # Get the first value of the 'dataset' column if it exists
            dataset_value = df_sub["dataset"].iloc[0] if "dataset" in df_sub.columns and not df_sub.empty else None

            # Mean of relevant numeric variables
            mean_values = df_sub.select_dtypes(include='number').mean(numeric_only=True).to_dict()

            dataset_summaries.append({
                "file": file,
                "dataset": dataset_value,
                "num_UUIDs": num_uuids,
                "avg_responses_per_UUID": round(avg_responses_per_uuid, 3),
                **mean_values
            })

    except Exception as e:
        print(f"Error processing {file}: {e}")

summary_df = pd.DataFrame(dataset_summaries)

print(f"Total unique subjects (UUIDs) across all studies: {len(all_uuids)}")
print(f"Total number of EMA responses (rows) across all studies: {total_emas}")

summary_df

Total unique subjects (UUIDs) across all studies: 843
Total number of EMA responses (rows) across all studies: 95864


,file,dataset,num_UUIDs,avg_responses_per_UUID,ANG_ES,HAP_ES,SAD_ES,RLX_ES,DIST_ES,SUPR_ES,REAP1_ES,REAP_ES,DEP_ES,CALM_ES
0,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,FEEL Study 1,179,165.179,15.537153,65.083708,18.441472,60.265296,51.794805,38.677952,42.268103,NaN,NaN,NaN
1,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Leuven emotions in daily life 2012,101,70.337,14.726914,57.076295,16.670467,57.861486,NaN,0.045186,NaN,0.040681,NaN,NaN
2,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Leuven 3-wave longitudinal study,202,174.277,11.071980,58.296103,11.934013,61.054340,22.655636,19.181854,NaN,NaN,11.683445,NaN
3,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Leuven emotions in daily life 2011,97,59.928,14.145192,56.138655,17.827112,58.226733,29.585068,23.745398,NaN,18.388268,16.869431,NaN
4,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Leuven emotion dynamics 2017,36,60.583,6.827144,58.657038,8.671252,57.217331,15.459422,NaN,NaN,NaN,NaN,NaN
5,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Everyday emotion regulation,50,51.920,1.712250,4.125193,NaN,3.998074,2.784669,2.465716,NaN,2.104777,1.905624,NaN
6,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,ACU emotions in daily life,74,62.122,13.177942,62.266478,16.981510,NaN,33.520992,21.366543,30.024799,NaN,NaN,56.043724
7,Z:\Projects\EMA_Project\Data\EMOTE\data_downlo...,Emotional events in daily life,104,84.635,9.981141,62.860600,11.474097,62.280618,34.219382,31.109066,NaN,33.845149,NaN,NaN


In [5]:
# Part 2: Get timeseries and scale to match
def rescale_to_1_5(df, columns):
    for col in columns:
        if col in df.columns:
            min_val, max_val = df[col].min(), df[col].max()
            if not (np.isclose(min_val, 1) and np.isclose(max_val, 5)):
                scaler = MinMaxScaler(feature_range=(1, 5))
                df[col] = scaler.fit_transform(df[[col]])
    return df

def special_round(avg, values):
    if avg > 3 and any(val >= 4.9 for val in values):
        return np.ceil(avg)
    elif avg < 3 and any(val <= 1.1 for val in values):
        return np.floor(avg)
    else:
        return round(avg)

regulation_vars = ["DIST_ES", "SUPR_ES", "REAP_ES"]
regulation_presence = {var: [] for var in regulation_vars}  # Track presence across files

# Store time series by dataset
participant_ts_by_file = {}

for summary in dataset_summaries:
    file = summary["file"]
    try:
        df = pd.read_csv(file)
        # Only use columns that exist in this dataset
        present_vars = [var for var in all_vars if var in df.columns]
        # If REAP1_ES is present but REAP_ES is not, rename it for consistency
        if "REAP1_ES" in present_vars and "REAP_ES" not in present_vars:
            df = df.rename(columns={"REAP1_ES": "REAP_ES"})
            present_vars = [v if v != "REAP1_ES" else "REAP_ES" for v in present_vars]
        # Only dropna for present variables
        df = df.dropna(subset=present_vars)
        # Only rescale numeric columns
        numeric_vars = [var for var in present_vars if pd.api.types.is_numeric_dtype(df[var])]
        df = rescale_to_1_5(df, numeric_vars)

        # Determine which optional variables to use
        pos_vars = ["HAP_ES"]
        if "RLX_ES" in df.columns:
            pos_vars.append("RLX_ES")
        elif "CALM_ES" in df.columns:
            pos_vars.append("CALM_ES")

        neg_vars = ["ANG_ES"]
        if "SAD_ES" in df.columns:
            neg_vars.append("SAD_ES")
        elif "DEP_ES" in df.columns:
            neg_vars.append("DEP_ES")

        # Group by UUID and extract time series
        ts_data = {}
        for uuid, group in df.groupby("UUID"):
            group = group.sort_values(by="Time_Local")

            pos_score = [round(val) for val in group[pos_vars].max(axis=1)]
            neg_score = [round(val) for val in group[neg_vars].max(axis=1)]

            # Initialize this UUID's time series entry
            ts_data[uuid] = {
                "Positive": pos_score,
                "Negative": neg_score,
            }

            # Add each regulation variable if it exists
            for var in regulation_vars:
                if var in df.columns:
                    regulation_presence[var].append(file)
                    ts_data[uuid][var] = [round(val) for val in group[var]]

        participant_ts_by_file[file] = ts_data

    except Exception as e:
        print(f"Error in Part 2 for {file}: {e}")

In [6]:
definition_transition_matrix = ["Transition", "Proportion"]
distance_measure = ["Pearson", "Spearman", "Euclid"]
comparison_method = ["Pairwise", "Leave-One-Subject-Out"]
control_for_entropy = ["Entropy", "No_Entropy"]
temporal_overlap = ["Start_Date", "Response_Window", "Date_and_Time", "No_Time_Control"]

In [7]:
# Part 3: Create Transition Matrices
for definition in definition_transition_matrix:
    for dataset_filename, uuid_dict in participant_ts_by_file.items():
        # Extract dataset identifier for saving
        dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]

        # Collect time series for each variable across UUIDs
        posAffTS = [ts["Positive"] for ts in uuid_dict.values()]
        negAffTS = [ts["Negative"] for ts in uuid_dict.values()]
        selfER_1TS = [ts["REAP_ES"] for ts in uuid_dict.values() if "REAP_ES" in ts]
        selfER_2TS = [ts["SUPR_ES"] for ts in uuid_dict.values() if "SUPR_ES" in ts]
        selfER_3TS = [ts["DIST_ES"] for ts in uuid_dict.values() if "DIST_ES" in ts]

        # If variables are missing, fill in empty lists for consistency
        def safe_fill(ts_list, n=len(posAffTS)):
            return ts_list if len(ts_list) == n else [[] for _ in range(n)]
        
        selfER_1TS = safe_fill(selfER_1TS)
        selfER_2TS = safe_fill(selfER_2TS)
        selfER_3TS = safe_fill(selfER_3TS)

        # Use your existing functions to compute transition or proportion matrices
        if definition == "Transition":
            PA_timeseries_list, PA_transition_matrices = indiv_ts_tmat(pd.DataFrame(posAffTS))
            NA_timeseries_list, NA_transition_matrices = indiv_ts_tmat(pd.DataFrame(negAffTS))
            RP_timeseries_list, RP_transition_matrices = indiv_ts_tmat(pd.DataFrame(selfER_1TS))
            SP_timeseries_list, SP_transition_matrices = indiv_ts_tmat(pd.DataFrame(selfER_2TS))
            DS_timeseries_list, DS_transition_matrices = indiv_ts_tmat(pd.DataFrame(selfER_3TS))
        elif definition == "Proportion":
            PA_timeseries_list, PA_transition_matrices = indiv_ts_pmat(pd.DataFrame(posAffTS))
            NA_timeseries_list, NA_transition_matrices = indiv_ts_pmat(pd.DataFrame(negAffTS))
            RP_timeseries_list, RP_transition_matrices = indiv_ts_pmat(pd.DataFrame(selfER_1TS))
            SP_timeseries_list, SP_transition_matrices = indiv_ts_pmat(pd.DataFrame(selfER_2TS))
            DS_timeseries_list, DS_transition_matrices = indiv_ts_pmat(pd.DataFrame(selfER_3TS))

        # Save transition matrices
        for name, matrices in zip(["PA", "NA", "RP", "SP", "DS"], 
                                  [PA_transition_matrices, NA_transition_matrices, 
                                   RP_transition_matrices, SP_transition_matrices, 
                                   DS_transition_matrices]):
            flattened_matrices = [matrix.ravel() for matrix in matrices]
            matrices_df = pd.DataFrame(flattened_matrices)
            matrices_df.to_csv(os.path.join(
                output_folder, f"{name}_transition_matrices_{dataset_id}_{definition}.csv"), index=False)

        # Save time series
        for name, timeseries in zip(["PA", "NA", "RP", "SP", "DS"], 
                                    [PA_timeseries_list, NA_timeseries_list, 
                                     RP_timeseries_list, SP_timeseries_list, 
                                     DS_timeseries_list]):
            timeseries_df = pd.DataFrame(timeseries)
            timeseries_df.to_csv(os.path.join(
                output_folder, f"{name}_timeseries_list_{dataset_id}_{definition}.csv"), index=False)


In [8]:
# Part 4: LOSO and Pairwise RDM for all comparison methods in each dataset
for definition in definition_transition_matrix:
    print(f"Processing definition: {definition}")

    for dataset_filename in participant_ts_by_file.keys():
        dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]

        try:
            def load_matrix(name):
                path = os.path.join(output_folder, f"{name}_transition_matrices_{dataset_id}_{definition}.csv")
                return pd.read_csv(path).values if os.path.exists(path) else None

            print(f"Loading matrices for dataset: {dataset_id}")
            PA_flat = load_matrix("PA")
            NA_flat = load_matrix("NA")
            RP_flat = load_matrix("RP")
            SP_flat = load_matrix("SP")
            DS_flat = load_matrix("DS")

            if any(x is None for x in [PA_flat, NA_flat, RP_flat, SP_flat, DS_flat]):
                print(f"Skipping {dataset_id} due to missing required matrices.")
                continue

            def reshape_matrices(flat): return [row.reshape(5, 5) for row in flat]

            PA = reshape_matrices(PA_flat)
            NA = reshape_matrices(NA_flat)
            RP = reshape_matrices(RP_flat)
            SP = reshape_matrices(SP_flat)
            DS = reshape_matrices(DS_flat)

            for measure in distance_measure:
                print(f"  Measure: {measure}")
                use = {"Pearson": "pearson", "Spearman": "spear", "Euclid": "euclid"}[measure]

                for comparison in comparison_method:
                    print(f"    Comparison: {comparison}")
                    comp_func = {"Pairwise": "compute_rsm", "Leave-One-Subject-Out": "loso_similarity_matrix"}[comparison]
                    func_name = f"{comp_func}_{use}"
                    print([k for k in globals().keys() if k.startswith("compute_rsm") or k.startswith("loso_similarity_matrix")])
                    print(str(globals()[func_name]))

                    # Compute similarity matrices
                    compute = lambda X: globals()[func_name](X)
                    PArsm = compute(PA)
                    NArsm = compute(NA)
                    RPrsm = compute(RP)
                    SPrsm = compute(SP)
                    DSrsm = compute(DS)

                    def transform(X):
                        if measure in ["Pearson", "Spearman"]:
                            return 1 - np.array(X)
                        else:
                            return 2 * np.array(X)

                    PArdm = transform(PArsm)
                    NArdm = transform(NArsm)
                    RPrdm = transform(RPrsm)
                    SPrdm = transform(SPrsm)
                    DSrdm = transform(DSrsm)

                    # if comparison == "Leave-One-Subject-Out":
                        # valid_mask = ~np.isnan(PArdm)
                        # valid_idx = np.where(valid_mask)[0]

                        # def subset(X):
                        #     return X[valid_idx]

                        # PArdm = subset(PArdm)
                        # NArdm = subset(NArdm)
                        # RPrdm = subset(RPrdm)
                        # SPrdm = subset(SPrdm)
                        # DSrdm = subset(DSrdm)

                        # np.save(os.path.join(output_folder, f"valid_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy"), valid_idx)

                    if comparison == "Pairwise": # Change back to elif if uncommented above
                        tril = lambda X: X[np.tril_indices_from(X, k=-1)]
                        PArdm = tril(PArdm)
                        NArdm = tril(NArdm)
                        RPrdm = tril(RPrdm)
                        SPrdm = tril(SPrdm)
                        DSrdm = tril(DSrdm)

                        matrix_shape = PArsm.shape
                        if matrix_shape is not None:
                            lower_triangle_indices = np.tril_indices(matrix_shape[0], k=-1)
                            np.save(os.path.join(output_folder, f"lower_triangle_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy"), lower_triangle_indices)

                        # valid_mask = ~np.isnan(PArdm)
                        # valid_idx = np.where(valid_mask)[0]

                        # def clean(X):
                        #     return X[valid_idx]

                        # PArdm = clean(PArdm)
                        # NArdm = clean(NArdm)
                        # RPrdm = clean(RPrdm)
                        # SPrdm = clean(SPrdm)
                        # DSrdm = clean(DSrdm)

                        # np.save(os.path.join(output_folder, f"valid_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy"), valid_idx)

                    # Save dissimilarity matrices
                    save_array = lambda name, X: np.save(
                        os.path.join(output_folder, f"{name}_{dataset_id}_{definition}_{measure}_{comparison}.npy"),
                        X.reshape(-1, 1))

                    save_array("PArdm", PArdm)
                    save_array("NArdm", NArdm)
                    save_array("RPrdm", RPrdm)
                    save_array("SPrdm", SPrdm)
                    save_array("DSrdm", DSrdm)

                    print(f"    Saved RDMs for {dataset_id} [{measure} | {comparison}]")
                    time.sleep(5)

            del PA, NA, RP, SP, DS
            gc.collect()

        except Exception as e:
            print(f"Error in {dataset_id} for {definition}: {e}")

Processing definition: Transition
Loading matrices for dataset: data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1
  Measure: Pearson
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_pearson at 0x000001F2420DD3A0>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1 [Pearson | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_pearson at 0x000001F2420DDBC0>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1 [Pearson | Leave-One-Subject-Out]
  Measure: Spearman
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_ma

z:\Projects\EMA_Project\Scripts\Multiverse_Analysis\tools.py:252: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, _ = pearsonr(subject_matrix, avg_matrix)


    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012 [Pearson | Leave-One-Subject-Out]
  Measure: Spearman
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_spear at 0x000001F2420DD800>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012 [Spearman | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_spear at 0x000001F2420DDC60>


z:\Projects\EMA_Project\Scripts\Multiverse_Analysis\tools.py:275: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, _ = spearmanr(subject_matrix, avg_matrix)


    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012 [Spearman | Leave-One-Subject-Out]
  Measure: Euclid
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_euclid at 0x000001F2420DD8A0>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012 [Euclid | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_euclid at 0x000001F2420DDD00>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012 [Euclid | Leave-One-Subject-Out]
Loading matrices for dataset: data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study
  Measure: Pear

...

    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study [Spearman | Leave-One-Subject-Out]
  Measure: Euclid
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_euclid at 0x000001F2420DD8A0>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study [Euclid | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_euclid at 0x000001F2420DDD00>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study [Euclid | Leave-One-Subject-Out]
Loading matrices for dataset: data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2011
  Measure: Pearson


z:\Projects\EMA_Project\Scripts\Multiverse_Analysis\tools.py:252: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, _ = pearsonr(subject_matrix, avg_matrix)


    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017 [Pearson | Leave-One-Subject-Out]
  Measure: Spearman
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_spear at 0x000001F2420DD800>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017 [Spearman | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_spear at 0x000001F2420DDC60>


z:\Projects\EMA_Project\Scripts\Multiverse_Analysis\tools.py:275: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, _ = spearmanr(subject_matrix, avg_matrix)


    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017 [Spearman | Leave-One-Subject-Out]
  Measure: Euclid
    Comparison: Pairwise
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function compute_rsm_euclid at 0x000001F2420DD8A0>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017 [Euclid | Pairwise]
    Comparison: Leave-One-Subject-Out
['compute_rsm_pearson', 'compute_rsm_spear', 'compute_rsm_euclid', 'loso_similarity_matrix_pearson', 'loso_similarity_matrix_spear', 'loso_similarity_matrix_euclid']
<function loso_similarity_matrix_euclid at 0x000001F2420DDD00>
    Saved RDMs for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017 [Euclid | Leave-One-Subject-Out]
Loading matrices for dataset: data_downloads_HLAJBYIHRK_2025-05-30Everyday_emotion_regulation
  Measure: Pearson
    Comparison: Pai

In [9]:
# Calculate entropy

for definition in definition_transition_matrix:
    for dataset_filename in participant_ts_by_file.keys():
        dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]

        # Load transition matrices
        def load_matrix(name):
            path = os.path.join(output_folder, f"{name}_transition_matrices_{dataset_id}_{definition}.csv")
            return pd.read_csv(path).values if os.path.exists(path) else None

        PA_flat = load_matrix("PA")
        NA_flat = load_matrix("NA")
        RP_flat = load_matrix("RP")
        SP_flat = load_matrix("SP")
        DS_flat = load_matrix("DS")

        if any(x is None for x in [PA_flat, NA_flat, RP_flat, SP_flat, DS_flat]):
            print(f"Skipping {dataset_id} for entropy due to missing matrices.")
            continue

        # Reshape to list of 5x5 matrices
        PA = [row.reshape(5, 5) for row in PA_flat]
        NA = [row.reshape(5, 5) for row in NA_flat]
        RP = [row.reshape(5, 5) for row in RP_flat]
        SP = [row.reshape(5, 5) for row in SP_flat]
        DS = [row.reshape(5, 5) for row in DS_flat]

        for measure in distance_measure:
            for comparison in comparison_method:
                # # Load valid indices
                # valid_idx_path = os.path.join(output_folder, f"valid_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy")
                # valid_indices = np.load(valid_idx_path) if os.path.exists(valid_idx_path) else None

                if comparison == "Leave-One-Subject-Out":
                    PAentropy = np.array(calculate_entropy_per_subject(PA))
                    NAentropy = np.array(calculate_entropy_per_subject(NA))
                    RPentropy = np.array(calculate_entropy_per_subject(RP))
                    SPentropy = np.array(calculate_entropy_per_subject(SP))
                    DSentropy = np.array(calculate_entropy_per_subject(DS))

                    # if valid_indices is not None:
                    #     PAentropy = PAentropy[valid_indices]
                    #     NAentropy = NAentropy[valid_indices]
                    #     RPentropy = RPentropy[valid_indices]
                    #     SPentropy = SPentropy[valid_indices]
                    #     DSentropy = DSentropy[valid_indices]

                    np.save(os.path.join(output_folder, f"PAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), PAentropy)
                    np.save(os.path.join(output_folder, f"NAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), NAentropy)
                    np.save(os.path.join(output_folder, f"RPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), RPentropy)
                    np.save(os.path.join(output_folder, f"SPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), SPentropy)
                    np.save(os.path.join(output_folder, f"DSentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), DSentropy)

                elif comparison == "Pairwise":
                    # Calculate pairwise entropy difference matrices
                    PAentropyMat = calculate_entropy_differences(PA)
                    NAentropyMat = calculate_entropy_differences(NA)
                    RPentropyMat = calculate_entropy_differences(RP)
                    SPentropyMat = calculate_entropy_differences(SP)
                    DSentropyMat = calculate_entropy_differences(DS)

                    # Load lower triangle indices
                    tril_path = os.path.join(output_folder, f"lower_triangle_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy")
                    lower_triangle_indices = np.load(tril_path, allow_pickle=True) if os.path.exists(tril_path) else None

                    print(f"Pairwise block for {dataset_id}: lower_triangle_indices exists? {lower_triangle_indices is not None}")
                    
                    if lower_triangle_indices is not None:
                        lower_triangle_indices = tuple(lower_triangle_indices)
                        PAentropy = PAentropyMat[lower_triangle_indices]
                        NAentropy = NAentropyMat[lower_triangle_indices]
                        RPentropy = RPentropyMat[lower_triangle_indices]
                        SPentropy = SPentropyMat[lower_triangle_indices]
                        DSentropy = DSentropyMat[lower_triangle_indices]

                        # if valid_indices is not None:
                        #     PAentropy = PAentropy[valid_indices]
                        #     NAentropy = NAentropy[valid_indices]
                        #     RPentropy = RPentropy[valid_indices]
                        #     SPentropy = SPentropy[valid_indices]
                        #     DSentropy = DSentropy[valid_indices]

                        np.save(os.path.join(output_folder, f"PAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), PAentropy)
                        np.save(os.path.join(output_folder, f"NAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), NAentropy)
                        np.save(os.path.join(output_folder, f"RPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), RPentropy)
                        np.save(os.path.join(output_folder, f"SPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), SPentropy)
                        np.save(os.path.join(output_folder, f"DSentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy"), DSentropy)

Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study: lower_triangle_indices exists? True
Pairwise block for data_downloads_HLAJBYIHRK_2025-05-3

In [10]:
# Custom functions for time due to different time formats than our dataset

from datetime import datetime

def EMOTE_date_differences_pairwise(date_list):
    """
    Calculate normalized date differences between all pairs of subjects for EMOTE data.
    date_list: list of strings in 'DD/MM/YYYY' format.
    Returns: symmetric matrix of date differences (in days).
    """
    dates = [datetime.strptime(d, "%d/%m/%Y") if not pd.isnull(d) else pd.NaT for d in date_list]
    ordinal_days = [d.toordinal() for d in dates]
    n = len(ordinal_days)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            diff = abs(ordinal_days[i] - ordinal_days[j])
            mat[i, j] = diff
            mat[j, i] = diff
    return mat

def EMOTE_date_differences_loso(date_list):
    """
    Calculate LOSO date differences for EMOTE data.
    date_list: list of strings in 'DD/MM/YYYY' format.
    Returns: list of LOSO differences (in days).
    """
    dates = [datetime.strptime(d, "%d/%m/%Y") if not pd.isnull(d) else pd.NaT for d in date_list]
    ordinal_days = [d.toordinal() for d in dates]
    n = len(ordinal_days)
    loso = []
    for i in range(n):
        others = ordinal_days[:i] + ordinal_days[i+1:]
        avg_others = np.mean(others)
        diff = abs(ordinal_days[i] - avg_others)
        loso.append(diff)
    return loso

def EMOTE_response_window_pairwise(df):
    """
    Calculate Euclidean distance matrix based on response windows for EMOTE data.
    df: DataFrame with 'UUID', 'Date_Local', 'Time_Local' columns.
    Returns: symmetric matrix of Euclidean distances.
    """
    # Combine date and time into datetime
    df = df.copy()
    df['DateTime'] = pd.to_datetime(df['Date_Local'] + ' ' + df['Time_Local'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df['TimeSlot'] = df['DateTime'].dt.strftime('%H:%M')
    # Use all observed time slots
    time_slots = sorted(df['TimeSlot'].dropna().unique())
    subject_time_tally = df.groupby(['UUID', 'TimeSlot']).size().unstack(fill_value=0)
    subject_time_tally = subject_time_tally.reindex(columns=time_slots, fill_value=0)
    tally_array = subject_time_tally.values
    n = tally_array.shape[0]
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            dist = euclidean(tally_array[i], tally_array[j])
            mat[i, j] = dist
            mat[j, i] = dist
    return mat

def EMOTE_response_window_loso(df):
    """
    Calculate LOSO Euclidean distances for response windows for EMOTE data.
    df: DataFrame with 'UUID', 'Date_Local', 'Time_Local' columns.
    Returns: list of LOSO distances.
    """
    df = df.copy()
    df['DateTime'] = pd.to_datetime(df['Date_Local'] + ' ' + df['Time_Local'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df['TimeSlot'] = df['DateTime'].dt.strftime('%H:%M')
    time_slots = sorted(df['TimeSlot'].dropna().unique())
    subject_time_tally = df.groupby(['UUID', 'TimeSlot']).size().unstack(fill_value=0)
    subject_time_tally = subject_time_tally.reindex(columns=time_slots, fill_value=0)
    tally_array = subject_time_tally.values
    n = tally_array.shape[0]
    loso = []
    for i in range(n):
        others = np.delete(tally_array, i, axis=0)
        avg_others = np.mean(others, axis=0)
        dist = euclidean(tally_array[i], avg_others)
        loso.append(dist)
    return loso

In [11]:
# Calculate time variables

for definition in definition_transition_matrix:
    for dataset_filename in participant_ts_by_file.keys():
        dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]
        # Load the original data for this dataset
        df = pd.read_csv(dataset_filename)
        if "UUID" not in df.columns or "Date_Local" not in df.columns or "Time_Local" not in df.columns:
            print(f"Skipping {dataset_id} for time variables due to missing columns.")
            continue

        # Drop rows with missing UUID, Date_Local, or Time_Local
        df = df.dropna(subset=["UUID", "Date_Local", "Time_Local"])

        # Get minimum date per UUID as string (for EMOTE functions)
        start_days = df.groupby("UUID")["Date_Local"].min().reset_index()
        start_days_l = start_days["Date_Local"].tolist()

        for measure in distance_measure:
            for comparison in comparison_method:
                # Load indices
                # valid_idx_path = os.path.join(output_folder, f"valid_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy")
                # valid_indices = np.load(valid_idx_path) if os.path.exists(valid_idx_path) else None

                tril_path = os.path.join(output_folder, f"lower_triangle_indices_{dataset_id}_{definition}_{measure}_{comparison}.npy")
                lower_triangle_indices = np.load(tril_path, allow_pickle=True) if os.path.exists(tril_path) else None

                # --- Start Date ---
                if comparison == "Leave-One-Subject-Out":
                    # LOSO: Use per-subject date differences
                    startDate_loso = EMOTE_date_differences_loso(start_days_l)
                    startDate_loso = np.array(startDate_loso)
                    # if valid_indices is not None:
                    #     startDate_loso = startDate_loso[valid_indices]
                    np.save(os.path.join(output_folder, f"startDate_{dataset_id}_{definition}_{measure}_{comparison}.npy"), startDate_loso)
                elif comparison == "Pairwise":
                    # Pairwise: Use pairwise date differences
                    if lower_triangle_indices is not None:
                        lower_triangle_indices = tuple(lower_triangle_indices)
                        normalized_date_diff_matrix = EMOTE_date_differences_pairwise(start_days_l)
                        startDate = np.array(normalized_date_diff_matrix)[lower_triangle_indices]
                        # if valid_indices is not None:
                        #     startDate = startDate[valid_indices]
                        np.save(os.path.join(output_folder, f"startDate_{dataset_id}_{definition}_{measure}_{comparison}.npy"), startDate)

                # --- Response Window ---
                if comparison == "Leave-One-Subject-Out":
                    responseWindowLoso = EMOTE_response_window_loso(df)
                    responseWindowLoso = np.array(responseWindowLoso)
                    # if valid_indices is not None:
                    #     responseWindowLoso = responseWindowLoso[valid_indices]
                    np.save(os.path.join(output_folder, f"responseWindow_{dataset_id}_{definition}_{measure}_{comparison}.npy"), responseWindowLoso)
                elif comparison == "Pairwise":
                    if lower_triangle_indices is not None:
                        lower_triangle_indices = tuple(lower_triangle_indices)
                        response_window_matrix = EMOTE_response_window_pairwise(df)
                        responseWindow = np.array(response_window_matrix)[lower_triangle_indices]
                        # if valid_indices is not None:
                        #     responseWindow = responseWindow[valid_indices]
                        np.save(os.path.join(output_folder, f"responseWindow_{dataset_id}_{definition}_{measure}_{comparison}.npy"), responseWindow)

In [12]:
# Helper function for this part, because some RDMs seem to be coming in as X, 1
def squeeze_array(arr):
    """
    Ensure array has shape (X,) instead of (X, 1)
    """
    if arr.ndim == 2 and arr.shape[1] == 1:
        return arr.squeeze(axis=1)
    return arr.squeeze() if arr.ndim > 1 and arr.shape[-1] == 1 else arr

In [13]:
for definition in definition_transition_matrix:
    for measure in distance_measure:
        for comparison in comparison_method:
            print(f"Processing {definition}, {measure}, {comparison}")
            
            # Initialize storage for concatenated data
            all_datasets = {}
            
            # Loop through all datasets to collect RDMs and covariates
            for dataset_filename in participant_ts_by_file.keys():
                dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]
                
                try:
                    # Load RDMs for this dataset
                    PArdm = squeeze_array(np.load(os.path.join(output_folder, f"PArdm_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    NArdm = squeeze_array(np.load(os.path.join(output_folder, f"NArdm_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    RPrdm = squeeze_array(np.load(os.path.join(output_folder, f"RPrdm_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    SPrdm = squeeze_array(np.load(os.path.join(output_folder, f"SPrdm_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    DSrdm = squeeze_array(np.load(os.path.join(output_folder, f"DSrdm_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                   
                    # Load covariates for this dataset
                    PAentropy = squeeze_array(np.load(os.path.join(output_folder, f"PAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    NAentropy = squeeze_array(np.load(os.path.join(output_folder, f"NAentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    RPentropy = squeeze_array(np.load(os.path.join(output_folder, f"RPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    SPentropy = squeeze_array(np.load(os.path.join(output_folder, f"SPentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    DSentropy = squeeze_array(np.load(os.path.join(output_folder, f"DSentropy_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    
                    startDate = squeeze_array(np.load(os.path.join(output_folder, f"startDate_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    responseWindow = squeeze_array(np.load(os.path.join(output_folder, f"responseWindow_{dataset_id}_{definition}_{measure}_{comparison}.npy")))
                    # if measure == "Pearson":
                    #     # Check shape
                    #     print("Shapes for dataset:", dataset_id)
                    #     print(PArdm.shape, NArdm.shape, RPrdm.shape, SPrdm.shape, DSrdm.shape)
                    #     print(PAentropy.shape, NAentropy.shape, RPentropy.shape, SPentropy.shape, DSentropy.shape)
                    #     print(startDate.shape, responseWindow.shape)
                    
                    # Store data for this dataset
                    all_datasets[dataset_id] = {
                        'PArdm': PArdm, 'NArdm': NArdm, 'RPrdm': RPrdm, 'SPrdm': SPrdm, 'DSrdm': DSrdm,
                        'PAentropy': PAentropy, 'NAentropy': NAentropy, 'RPentropy': RPentropy, 
                        'SPentropy': SPentropy, 'DSentropy': DSentropy,
                        'startDate': startDate, 'responseWindow': responseWindow
                    }
                    
                except Exception as e:
                    print(f"Could not load data for {dataset_id}: {e}")
                    continue
            
            # Identify datasets with all three core strategies (for joint regression)
            datasets_with_all_strategies = []
            for dataset_id, data in all_datasets.items():
                if all(len(data[var]) > 0 for var in ['RPrdm', 'SPrdm', 'DSrdm']):
                    datasets_with_all_strategies.append(dataset_id)
            
            print(f"Datasets with all strategies: {datasets_with_all_strategies}")
            
            for entropy in control_for_entropy:
                for time in temporal_overlap:
                    
                    # === JOINT REGRESSION (datasets with all three strategies) ===
                    if datasets_with_all_strategies:
                        print(f"Joint regression with {len(datasets_with_all_strategies)} datasets")
                        
                        # Concatenate data across datasets with all strategies
                        joint_data = {}
                        joint_dataset_ids = []
                        for ds in datasets_with_all_strategies:
                            n = len(all_datasets[ds]['RPrdm'])
                            joint_dataset_ids.extend([ds] * n)
                        for var in ['PArdm', 'NArdm', 'RPrdm', 'SPrdm', 'DSrdm', 
                                   'PAentropy', 'NAentropy', 'RPentropy', 'SPentropy', 'DSentropy',
                                   'startDate', 'responseWindow']:
                            joint_data[var] = np.concatenate([all_datasets[ds][var] for ds in datasets_with_all_strategies])
                        joint_data['dataset_id'] = np.array(joint_dataset_ids)
                        
                        # Set up covariates and labels
                        core_covariates = [joint_data['RPrdm'], joint_data['SPrdm'], joint_data['DSrdm']]
                        core_labels = ["RP", "SP", "DS"]
                        outcomes = {"PA": joint_data['PArdm'], "NA": joint_data['NArdm']}

                        optional_covariates = []
                        optional_labels = []
                        if time == "Start_Date" or time == "Date_and_Time":
                            optional_covariates.append(joint_data['startDate'])
                            optional_labels.append("startDate")
                                    
                        if time == "Response_Window" or time == "Date_and_Time":
                            optional_covariates.append(joint_data['responseWindow'])
                            optional_labels.append("responseWindow")

                        entropy_covariates = {
                            "PA": [joint_data['PAentropy'], joint_data['RPentropy'], joint_data['SPentropy'], joint_data['DSentropy']],
                            "NA": [joint_data['NAentropy'], joint_data['RPentropy'], joint_data['SPentropy'], joint_data['DSentropy']],  
                        }
                        entropy_labels = ["PAentropy", "RPentropy", "SPentropy", "DSentropy"]
                        
                        # Run joint regression
                        for outcome_label, outcome_data in outcomes.items():
                            X = core_covariates + optional_covariates[:]
                            labels = core_labels + optional_labels[:]
                                    
                            if entropy == "Entropy":
                                X += entropy_covariates[outcome_label]
                                labels += entropy_labels
                            
                            # Remove missing data
                            X_mat_temp = np.column_stack(X)
                            print("Raw shape:", X_mat_temp.shape, outcome_data.shape)
                            mask = ~np.isnan(outcome_data) & ~np.isnan(X_mat_temp).any(axis=1)
                            # Check how many are being dropped in each column and print labels
                            print("Missing data per column:")
                            for i, lbl in enumerate(labels):
                                missing_count = np.isnan(X_mat_temp[:, i]).sum()
                                print(f"  {lbl}: {missing_count} missing")
                            outcome_clean = outcome_data[mask]
                            X_clean = X_mat_temp[mask]
                            print("Cleaned shapes:", outcome_clean.shape, X_clean.shape)
                            dataset_id_clean = joint_data['dataset_id'][mask]
                            
                            if len(outcome_clean) > 0:
                                X_mat = sm.add_constant(X_clean)
                                variable_labels = {f"x{i+1}": lbl for i, lbl in enumerate(labels)}
                                variable_labels = {'const': 'Intercept', **variable_labels}
                                
                                # OLS with clustered SEs (only dataset_id)
                                model = sm.OLS(outcome_clean, X_mat)
                                results = model.fit(
                                    cov_type='cluster',
                                    cov_kwds={'groups': dataset_id_clean}
                                )
                                
                                param_names = results.model.exog_names
                                df = pd.DataFrame({
                                    "Variable": [variable_labels.get(name, name) for name in param_names],
                                    "Parameter": results.params,
                                    "Std_Err": results.bse,
                                    "P_value": results.pvalues,
                                    "Observations": [results.nobs] * len(results.params)
                                })

                                for label in core_labels:
                                    specification = f"JOINT_{definition}_{measure}_{comparison}_{entropy}_{time}"
                                    filename = f"{outcome_label}_{label}_result_{specification}.csv"
                                    df_label = df[df["Variable"] == label]
                                    if not df_label.empty:
                                        df_label.to_csv(os.path.join(results_folder, filename), index=False)
                    
                    # === SINGLE-PREDICTOR REGRESSIONS ===
                    for strategy in ["RP", "SP", "DS"]:
                        # Find datasets that have this strategy
                        datasets_with_strategy = []
                        for dataset_id, data in all_datasets.items():
                            if np.sum(~np.isnan(data[f'{strategy}rdm'])) > 0:
                                datasets_with_strategy.append(dataset_id)
                        
                        if datasets_with_strategy:
                            print(f"Single {strategy} regression with {len(datasets_with_strategy)} datasets")
                            
                            # Concatenate data across datasets with this strategy
                            single_data = {}
                            single_dataset_ids = []
                            for ds in datasets_with_strategy:
                                n = len(all_datasets[ds][f'{strategy}rdm'])
                                single_dataset_ids.extend([ds] * n)
                            for var in ['PArdm', 'NArdm', f'{strategy}rdm', 
                                       'PAentropy', 'NAentropy', f'{strategy}entropy',
                                       'startDate', 'responseWindow']:
                                single_data[var] = np.concatenate([all_datasets[ds][var] for ds in datasets_with_strategy])
                            single_data['dataset_id'] = np.array(single_dataset_ids)
                            
                            outcomes = {"PA": single_data['PArdm'], "NA": single_data['NArdm']}
                            
                            for outcome_label, outcome_data in outcomes.items():
                                # Start with the single core predictor
                                X_single = [single_data[f'{strategy}rdm']]
                                labels_single = [strategy]

                                # Add optional covariates
                                optional_covariates_single = []
                                optional_labels_single = []
                                if time == "Start_Date" or time == "Date_and_Time":
                                    optional_covariates_single.append(single_data['startDate'])
                                    optional_labels_single.append("startDate")
                                        
                                if time == "Response_Window" or time == "Date_and_Time":
                                    optional_covariates_single.append(single_data['responseWindow'])
                                    optional_labels_single.append("responseWindow")

                                X_single += optional_covariates_single
                                labels_single += optional_labels_single

                                # Add entropy covariates if applicable
                                if entropy == "Entropy":
                                    entropy_single = [single_data[f'{outcome_label}entropy'], single_data[f'{strategy}entropy']]
                                    X_single += entropy_single
                                    labels_single += [f"{outcome_label}entropy", f"{strategy}entropy"]

                                # Remove missing data
                                X_mat_temp = np.column_stack(X_single)
                                mask = ~np.isnan(outcome_data) & ~np.isnan(X_mat_temp).any(axis=1)
                                outcome_clean = outcome_data[mask]
                                X_clean = X_mat_temp[mask]
                                dataset_id_clean = single_data['dataset_id'][mask]
                                
                                if len(outcome_clean) > 0:
                                    # Stack into design matrix
                                    X_single_mat = sm.add_constant(X_clean)
                                    variable_labels_single = {f"x{i+1}": lbl for i, lbl in enumerate(labels_single)}
                                    variable_labels_single = {'const': 'Intercept', **variable_labels_single}

                                    # OLS with clustered SEs (only dataset_id)
                                    model_single = sm.OLS(outcome_clean, X_single_mat)
                                    results_single = model_single.fit(
                                        cov_type='cluster',
                                        cov_kwds={'groups': dataset_id_clean}
                                    )

                                    param_names_single = results_single.model.exog_names
                                    df_single = pd.DataFrame({
                                        "Variable": [variable_labels_single.get(name, name) for name in param_names_single],
                                        "Parameter": results_single.params,
                                        "Std_Err": results_single.bse,
                                        "P_value": results_single.pvalues,
                                        "Observations": [results_single.nobs] * len(results_single.params)
                                    })

                                    specification = f"SINGLE_{strategy}_{definition}_{measure}_{comparison}_{entropy}_{time}"
                                    filename_single = f"{outcome_label}_{strategy}_single_result_{specification}.csv"
                                    if not df_single.empty:
                                        df_single.to_csv(os.path.join(results_folder, filename_single), index=False)

Processing Transition, Pearson, Pairwise
Datasets with all strategies: ['data_downloads_HLAJBYIHRK_2025-05-30FEEL_Study_1', 'data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2012', 'data_downloads_HLAJBYIHRK_2025-05-30Leuven_3-wave_longitudinal_study', 'data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotions_in_daily_life_2011', 'data_downloads_HLAJBYIHRK_2025-05-30Leuven_emotion_dynamics_2017', 'data_downloads_HLAJBYIHRK_2025-05-30Everyday_emotion_regulation', 'data_downloads_HLAJBYIHRK_2025-05-30ACU_emotions_in_daily_life', 'data_downloads_HLAJBYIHRK_2025-05-30Emotional_events_in_daily_life']
Joint regression with 8 datasets
Raw shape: (55850, 8) (55850,)
Missing data per column:
  RP: 20931 missing
  SP: 630 missing
  DS: 5050 missing
  startDate: 0 missing
  PAentropy: 0 missing
  RPentropy: 0 missing
  SPentropy: 0 missing
  DSentropy: 0 missing
Cleaned shapes: (29869,) (29869, 8)
Raw shape: (55850, 8) (55850,)
Missing data per column:
  RP: 20931 missing
  SP: 630 mi

In [14]:
from scipy import stats

def cluster_robust_se(model_results, cluster_var):
    """Compute cluster-robust standard errors manually"""
    from statsmodels.stats.sandwich_covariance import cov_cluster
    cov_cluster_robust = cov_cluster(model_results, cluster_var)
    return np.sqrt(np.diag(cov_cluster_robust))

# --- Build long dataframe for all datasets, keeping track of dataset_id and UUID ---
all_rows = []
for dataset_filename, uuid_dict in participant_ts_by_file.items():
    dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]
    for uuid, ts in uuid_dict.items():
        n = len(ts["Positive"])
        row = {
            "dataset_id": [dataset_id]*n,
            "UUID": [uuid]*n,
            "PA": ts["Positive"],
            "NA": ts["Negative"],
        }
        # Add only present strategies
        for strat, strat_col in zip(["REAP_ES", "SUPR_ES", "DIST_ES"], ["RP", "SP", "DS"]):
            if strat in ts:
                row[strat_col] = ts[strat]
        all_rows.append(pd.DataFrame(row))
df_all = pd.concat(all_rows, ignore_index=True)

# --- Concurrent regressions ---
results = []

# Determine which strategies are present in the combined data
present_strats = [col for col in ["RP", "SP", "DS"] if col in df_all.columns and df_all[col].notnull().any()]

# Joint regression: only if all strategies are present
if all(s in present_strats for s in ["RP", "SP", "DS"]):
    joint_mask = df_all[["PA", "NA", "RP", "SP", "DS", "UUID", "dataset_id"]].notnull().all(axis=1)
    df_joint = df_all[joint_mask].copy()
    for outcome in ["PA", "NA"]:
        X = df_joint[["RP", "SP", "DS"]].values
        y = df_joint[outcome].values
        X_mat = sm.add_constant(X)
        variable_labels = {'const': 'Intercept', 'x1': 'RP', 'x2': 'SP', 'x3': 'DS'}
        
        # Fit basic OLS
        model = sm.OLS(y, X_mat)
        results_joint = model.fit()
        
        # Compute cluster-robust SEs (cluster by dataset_id)
        cluster_se = cluster_robust_se(results_joint, df_joint['dataset_id'])
        
        # Compute p-values with cluster-robust SEs
        t_stats = results_joint.params / cluster_se
        p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=results_joint.df_resid))
        
        param_names = results_joint.model.exog_names
        summary_df = pd.DataFrame({
            "Analysis": "Concurrent",
            "Predictor_Set": "All",
            "Outcome": outcome,
            "Predictor": [variable_labels.get(name, name) for name in param_names],
            "Coefficient": results_joint.params,
            "Std_Err": cluster_se,
            "p-value": p_values,
            "Observations": [results_joint.nobs] * len(results_joint.params)
        })
        summary_df.to_csv(os.path.join(results_folder, f"concurrent_CLUSTERED_{outcome}_all.csv"), index=False)
        results.append(summary_df)

# Individual regressions: maximize data for each predictor
for strat in present_strats:
    mask = df_all[["PA", "NA", strat, "UUID", "dataset_id"]].notnull().all(axis=1)
    df_strat = df_all[mask].copy()
    if df_strat.empty:
        continue
    for outcome in ["PA", "NA"]:
        X = df_strat[[strat]].values
        y = df_strat[outcome].values
        X_mat = sm.add_constant(X)
        variable_labels = {'const': 'Intercept', 'x1': strat}
        
        # Fit basic OLS
        model = sm.OLS(y, X_mat)
        results_strat = model.fit()
        
        # Compute cluster-robust SEs (cluster by dataset_id)
        cluster_se = cluster_robust_se(results_strat, df_strat['dataset_id'])
        
        # Compute p-values with cluster-robust SEs
        t_stats = results_strat.params / cluster_se
        p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=results_strat.df_resid))
        
        param_names = results_strat.model.exog_names
        summary_df = pd.DataFrame({
            "Analysis": "Concurrent",
            "Predictor_Set": strat,
            "Outcome": outcome,
            "Predictor": [variable_labels.get(name, name) for name in param_names],
            "Coefficient": results_strat.params,
            "Std_Err": cluster_se,
            "p-value": p_values,
            "Observations": [results_strat.nobs] * len(results_strat.params)
        })
        summary_df.to_csv(os.path.join(results_folder, f"concurrent_CLUSTERED_{outcome}_{strat}.csv"), index=False)
        results.append(summary_df)

# --- Prospective regressions ---
lag_rows = []
for dataset_filename, uuid_dict in participant_ts_by_file.items():
    dataset_id = os.path.splitext(os.path.basename(dataset_filename))[0]
    for uuid, ts in uuid_dict.items():
        n = len(ts["Positive"])
        if n < 2:
            continue
        row = {
            "dataset_id": [dataset_id]*(n-1),
            "UUID": [uuid]*(n-1),
            "PA_t1": ts["Positive"][1:],
            "NA_t1": ts["Negative"][1:],
        }
        for strat, strat_col in zip(["REAP_ES", "SUPR_ES", "DIST_ES"], ["RP", "SP", "DS"]):
            if strat in ts:
                row[strat_col] = ts[strat][:-1]
        lag_rows.append(pd.DataFrame(row))
if lag_rows:
    lagged_all = pd.concat(lag_rows, ignore_index=True)
    present_strats_lag = [col for col in ["RP", "SP", "DS"] if col in lagged_all.columns and lagged_all[col].notnull().any()]
    # Joint regression: only if all strategies are present
    if all(s in present_strats_lag for s in ["RP", "SP", "DS"]):
        joint_mask = lagged_all[["PA_t1", "NA_t1", "RP", "SP", "DS", "UUID", "dataset_id"]].notnull().all(axis=1)
        lagged_joint = lagged_all[joint_mask].copy()
        for outcome in ["PA", "NA"]:
            ycol = "PA_t1" if outcome == "PA" else "NA_t1"
            X = lagged_joint[["RP", "SP", "DS"]].values
            y = lagged_joint[ycol].values
            X_mat = sm.add_constant(X)
            variable_labels = {'const': 'Intercept', 'x1': 'RP', 'x2': 'SP', 'x3': 'DS'}
            
            # Fit basic OLS
            model = sm.OLS(y, X_mat)
            results_joint = model.fit()
            
            # Compute cluster-robust SEs (cluster by dataset_id)
            cluster_se = cluster_robust_se(results_joint, lagged_joint['dataset_id'])
            
            # Compute p-values with cluster-robust SEs
            t_stats = results_joint.params / cluster_se
            p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=results_joint.df_resid))
            
            param_names = results_joint.model.exog_names
            summary_df = pd.DataFrame({
                "Analysis": "Prospective",
                "Predictor_Set": "All",
                "Outcome": outcome,
                "Predictor": [variable_labels.get(name, name) for name in param_names],
                "Coefficient": results_joint.params,
                "Std_Err": cluster_se,
                "p-value": p_values,
                "Observations": [results_joint.nobs] * len(results_joint.params)
            })
            summary_df.to_csv(os.path.join(results_folder, f"prospective_CLUSTERED_{outcome}_all.csv"), index=False)
            results.append(summary_df)
    # Individual regressions
    for strat in present_strats_lag:
        mask = lagged_all[["PA_t1", "NA_t1", strat, "UUID", "dataset_id"]].notnull().all(axis=1)
        lagged_strat = lagged_all[mask].copy()
        if lagged_strat.empty:
            continue
        for outcome in ["PA", "NA"]:
            ycol = "PA_t1" if outcome == "PA" else "NA_t1"
            X = lagged_strat[[strat]].values
            y = lagged_strat[ycol].values
            X_mat = sm.add_constant(X)
            variable_labels = {'const': 'Intercept', 'x1': strat}
            
            # Fit basic OLS
            model = sm.OLS(y, X_mat)
            results_strat = model.fit()
            
            # Compute cluster-robust SEs (cluster by dataset_id)
            cluster_se = cluster_robust_se(results_strat, lagged_strat['dataset_id'])
            
            # Compute p-values with cluster-robust SEs
            t_stats = results_strat.params / cluster_se
            p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=results_strat.df_resid))
            
            param_names = results_strat.model.exog_names
            summary_df = pd.DataFrame({
                "Analysis": "Prospective",
                "Predictor_Set": strat,
                "Outcome": outcome,
                "Predictor": [variable_labels.get(name, name) for name in param_names],
                "Coefficient": results_strat.params,
                "Std_Err": cluster_se,
                "p-value": p_values,
                "Observations": [results_strat.nobs] * len(results_strat.params)
            })
            summary_df.to_csv(os.path.join(results_folder, f"prospective_CLUSTERED_{outcome}_{strat}.csv"), index=False)
            results.append(summary_df)

print("All first-order regressions for EMOTE (across datasets, dynamic, clustered SEs by dataset) completed.")

All first-order regressions for EMOTE (across datasets, dynamic, clustered SEs by dataset) completed.


In [17]:
# Code for collecting multiverse results to graph
# Setup
graph_folder = r"Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Combined_Results"
os.makedirs(graph_folder, exist_ok=True)

results_folder = r"Z:\Projects\EMA_Project\Scripts\Output\EMOTE_Results"
csv_files = [f for f in os.listdir(results_folder) if f.endswith(".csv")]
dataframes = {}
for file in csv_files:
    try:
        file_path = os.path.join(results_folder, file)
        dataframes[file] = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Function to extract parameter and CI data
def get_graph_data(outcome_var, predictor_var, dataframes, mode):
    parameter_data = []
    lower_data = []
    upper_data = []
    predictor_labels = []

    suffix = "_single_" if mode == "single" else "_result_"

    for file, data in dataframes.items():
        if f"{outcome_var}_{predictor_var}{suffix}" in file:
            if all(col in data.columns for col in ["Variable", "Parameter", "Std_Err", "Observations"]):
                predictor_row = data[data["Variable"] == predictor_var]
                if not predictor_row.empty:
                    try:
                        parameter_value = predictor_row.iloc[0]["Parameter"]
                        std_error = predictor_row.iloc[0]["Std_Err"]
                        observations = predictor_row.iloc[0]["Observations"]
                        alpha = 0.05 / (np.sqrt(observations / 100))
                        z_score = norm.ppf(1 - (alpha / 2))
                        lower_bound = parameter_value - (z_score * std_error)
                        upper_bound = parameter_value + (z_score * std_error)

                        parameter_data.append(parameter_value)
                        lower_data.append(lower_bound)
                        upper_data.append(upper_bound)
                        predictor_labels.append(file)
                    except Exception as e:
                        print(f"Error processing row in {file}: {e}")

    if parameter_data:
        sorted_indices = np.argsort(parameter_data)
        return (
            np.array(parameter_data)[sorted_indices],
            np.array(lower_data)[sorted_indices],
            np.array(upper_data)[sorted_indices],
            np.array(predictor_labels)[sorted_indices],
        )
    else:
        return np.array([]), np.array([]), np.array([]), np.array([])

# Loop over all combinations (adjusted for EMOTE)
for o_var in ["PA", "NA"]:
    for p_var in ["RP", "SP", "DS"]:
        for mode in ["multi", "single"]:
            parameter_data, lower_data, upper_data, predictor_labels = get_graph_data(o_var, p_var, dataframes, mode=mode)
            if parameter_data.size > 0:
                suffix = "single" if mode == "single" else "multi"
                df = pd.DataFrame({
                    "Parameter": parameter_data,
                    "Lower": lower_data,
                    "Upper": upper_data,
                    "Specification": predictor_labels
                })
                csv_path = os.path.join(graph_folder, f"{o_var}_{p_var}_{suffix}.csv")
                df.to_csv(csv_path, index=False)